# Phi-2 SAE + Field View — exp_001_sae_phi2

**Base76 Research Lab** · Cross-model replication of the Field View reliability signal.

Trains a Sparse Autoencoder on `microsoft/phi-2` (layer 16) and runs three Field View scenarios:
- `math_det`: deterministic / anchored inference state
- `analogy_reason`: structured reasoning state  
- `hallucination`: hallucination-prone / degenerate state

**Requirements:** GPU runtime. Go to `Runtime → Change runtime type → T4 GPU`.

---
## Setup options
**Option A (recommended):** Cell below clones automatically from GitHub.  
**Option B:** Mount Google Drive — put project at `MyDrive/Mechanistic-Interpretability/`.  
**Option C:** Upload zip → `!unzip phi2_project.zip` → skip clone cell.


In [ ]:
# 0. Verify GPU
!nvidia-smi

In [ ]:
# 1. Install dependencies
!pip install -q transformers accelerate torch einops matplotlib pandas

In [ ]:
# 2. Get project (Option A: GitHub) — Option B: uncomment Drive lines
import os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/base76-research-lab/Mechanistic-Interpretability.git'
ROOT = pathlib.Path('Mechanistic-Interpretability')

# Option B — Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# ROOT = pathlib.Path('/content/drive/MyDrive/Mechanistic-Interpretability')

if not ROOT.exists():
    print(f'Cloning {REPO_URL} ...')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)])
    print('Done.')
else:
    print(f'Using existing: {ROOT.resolve()}')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
print('CWD:', os.getcwd())


In [ ]:
# 3. Train SAE on Phi-2 layer 16 (~5-10 min on T4)
!python3 scripts/run_sae.py \
    --model microsoft/phi-2 \
    --layer 16 \
    --dict-size 512 \
    --steps 200 \
    --lr 1e-3 --l1 1e-3 \
    --layernorm \
    --suffix _phi2 \
    --device cuda


In [ ]:
# 4. Field View — three scenarios
import subprocess

COMMON = [
    '--model', 'microsoft/phi-2',
    '--layer', '16',
    '--sae_state', 'experiments/exp_001_sae_phi2/sae_weights.pt',
    '--mode', 'pc2',
    '--topk', '10',
    '--device', 'cuda',
]

SCENARIOS = [
    ('math_det_phi2',         '2 + 2 ='),
    ('analogy_reason_phi2',   'king is to queen as man is to'),
    ('hallucination_phi2',    'who was the president of france in 1200?'),
]

for scenario, prompt in SCENARIOS:
    print(f'\n── {scenario} ──')
    cmd = ['python3', 'scripts/run_field_view_logged.py',
           '--scenario', scenario, '--prompt', prompt] + COMMON
    subprocess.run(cmd, check=True)


In [ ]:
# 5. Results table + GPT-2 comparison
import json, pathlib, pandas as pd

GPT2_REF = [
    {'scenario': 'math_det',       'model': 'GPT-2 small', 'layer': 5,  'H': 5.123, 'gap': 0.691, 'risk': 0.549, 'state': 'A — Anchored'},
    {'scenario': 'analogy_reason', 'model': 'GPT-2 small', 'layer': 5,  'H': 6.467, 'gap': 3.951, 'risk': 1.000, 'state': 'B — Reasoning'},
    {'scenario': 'hallucination',  'model': 'GPT-2 small', 'layer': 5,  'H': 6.150, 'gap': 6.814, 'risk': 1.000, 'state': 'D — Hallucination-prone'},
]

runs_dir = pathlib.Path('experiments/exp_001_sae_phi2/runs')
phi2_rows = []
for f in sorted(runs_dir.glob('*.json')):
    d = json.loads(f.read_text())
    m = d.get('metrics', {})
    sc = d.get('scenario', '').replace('_phi2', '')
    phi2_rows.append({
        'scenario': sc,
        'model': 'Phi-2',
        'layer': 16,
        'H': round(m.get('logit_entropy', float('nan')), 3),
        'gap': round(m.get('gap_state_to_candidates', float('nan')), 3),
        'risk': round(m.get('risk_score', float('nan')), 3),
        'state': '?',
    })

df = pd.DataFrame(GPT2_REF + phi2_rows).sort_values(['scenario', 'model'])
print(df.to_string(index=False))
print('\nHypothesis check: does Phi-2 show H↑ gap↑↑ for hallucination vs reasoning?')


In [ ]:
# 6. Scatter plots
import json, pathlib, matplotlib.pyplot as plt, matplotlib.patches as mpatches

COLORS = {
    'math_det_phi2':       '#22c55e',
    'analogy_reason_phi2': '#38bdf8',
    'hallucination_phi2':  '#ef4444',
}
STATE_LABELS = {
    'math_det_phi2':       'A — Anchored',
    'analogy_reason_phi2': 'B — Reasoning',
    'hallucination_phi2':  'D — Hallucination-prone',
}

runs_dir = pathlib.Path('experiments/exp_001_sae_phi2/runs')
pts = []
for f in sorted(runs_dir.glob('*.json')):
    d = json.loads(f.read_text())
    m, cf, sc = d.get('metrics', {}), d.get('candidate_front', {}), d.get('scenario', '')
    pts.append({'sc': sc, 'H': m.get('logit_entropy'), 'gap': m.get('gap_state_to_candidates'),
                'degen': cf.get('degeneracy_ratio_topk')})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
BG = '#0f172a'
fig.patch.set_facecolor(BG)
for ax in (ax1, ax2):
    ax.set_facecolor(BG)
    ax.tick_params(colors='#94a3b8'); ax.xaxis.label.set_color('#94a3b8'); ax.yaxis.label.set_color('#94a3b8')
    ax.title.set_color('#f1f5f9')
    for sp in ax.spines.values(): sp.set_edgecolor('#334155')
    ax.grid(color='#1e293b', linestyle='--', alpha=0.6)

for p in pts:
    c = COLORS.get(p['sc'], '#94a3b8')
    lbl = STATE_LABELS.get(p['sc'], p['sc'])
    if p['H'] and p['gap']:
        ax1.scatter(p['H'], p['gap'], color=c, s=130, zorder=5)
        ax1.annotate(p['sc'].replace('_phi2',''), (p['H'], p['gap']), xytext=(5,5),
                     textcoords='offset points', color=c, fontsize=9)
    if p['degen'] and p['gap']:
        ax2.scatter(p['degen'], p['gap'], color=c, s=130, zorder=5)
        ax2.annotate(p['sc'].replace('_phi2',''), (p['degen'], p['gap']), xytext=(5,5),
                     textcoords='offset points', color=c, fontsize=9)

ax1.set_xlabel('Logit Entropy (H)'); ax1.set_ylabel('State-Candidate Gap')
ax1.set_title('Gap vs Entropy — Phi-2 layer 16')
ax2.set_xlabel('Degeneracy Ratio top-k'); ax2.set_ylabel('State-Candidate Gap')
ax2.set_title('Gap vs Degeneracy — Phi-2 layer 16')

handles = [mpatches.Patch(color=c, label=STATE_LABELS[s]) for s, c in COLORS.items()]
ax1.legend(handles=handles, facecolor='#1e293b', labelcolor='#f1f5f9', fontsize=9)

out = pathlib.Path('reports/figures/phi2_field_view.png')
out.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved:', out)


In [ ]:
# 7. Package results for download
!zip -r phi2_results.zip experiments/exp_001_sae_phi2 reports/figures/phi2_field_view.png 2>/dev/null
try:
    from google.colab import files
    files.download('phi2_results.zip')
    print('Download started.')
except ImportError:
    print('phi2_results.zip ready in working directory.')
